# 002 · Bronze: landing (CSV) → Delta

**Trabalho 3 — Lakehouse com Databricks e arquitetura medalhão** · Autores: **Murilo Mandelli** e **Augusto Benedetti**

Lê **todos** os CSV do volume `landing/dados` e grava cada um como tabela **Delta** no schema
**`bronze`**. A bronze é uma **cópia fiel** da landing: nenhum dado é corrigido aqui (os problemas
de qualidade continuam lá). Só acrescentamos **metadados de ingestão**:

- `data_hora_bronze` — quando o dado foi processado;
- `nome_arquivo` — de qual arquivo da landing ele veio.

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

CAMINHO_LANDING = "/Volumes/workspace/landing/dados"

arquivos = sorted(f.name for f in dbutils.fs.ls(CAMINHO_LANDING) if f.name.endswith(".csv"))
print(f"{len(arquivos)} arquivos na landing:", arquivos)

## Ler cada CSV e gravar como tabela Delta gerenciada

`saveAsTable("bronze.<tabela>")` cria uma tabela **gerenciada** (MANAGED): o Databricks cuida dos
dados e dos metadados no Unity Catalog.

In [ ]:
for arquivo in arquivos:
    tabela = arquivo.removesuffix(".csv")
    df = (
        spark.read.option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{CAMINHO_LANDING}/{arquivo}")
        .withColumn("data_hora_bronze", current_timestamp())
        .withColumn("nome_arquivo", lit(arquivo))
    )
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"bronze.{tabela}"))
    print(f"bronze.{tabela:<12} {df.count():>6} registros")

In [ ]:
%sql
SHOW TABLES IN bronze

Os problemas da origem continuam na bronze (ela é uma cópia fiel):

In [ ]:
%sql
SELECT id_cliente, nome, email, cpf, cidade, uf, nome_arquivo
FROM bronze.cliente
WHERE cpf LIKE '%.%' OR email <> lower(email) OR cidade IS NULL
LIMIT 10

In [ ]:
%sql
DESCRIBE EXTENDED bronze.cliente